# Run_Smoke_All · smoke-test notebooks 01 → 10 in ONE Colab runtime

Colab's free tier allows one session at a time and every notebook asks for Drive access in its own runtime.
This driver mounts Drive **once**, then executes each notebook from `MyDrive/MAVREC/Notebooks/` in a fresh kernel
on this same T4 VM with `nbclient`:

* the notebooks run **unchanged**, except that an in-memory copy gets `SMOKE_TEST = True` (the files on Drive keep `False`)
  and a first cell that reuses this runtime's Drive mount instead of asking again;
* stops at the first failing cell and prints its traceback;
* every executed copy (all outputs) is written to `Final_Project_Results/smoke/executed/<notebook>.ipynb` and
  zipped for download (PDF records).

Needs notebook 00 to have run for real first. Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.


In [5]:
# 1 · Mount Drive once + same package pins as every notebook
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.173", "transformers==5.18.0",
                "pycocotools>=2.0.8", "timm>=1.0.15", "tabulate", "tensorboard", "seaborn", "nbclient", "nbformat"], check=True)
NB_DIR = "/content/drive/MyDrive/MAVREC/Notebooks"
OUT_DIR = "/content/drive/MyDrive/MAVREC/Final_Project_Results/smoke/executed"
os.makedirs(OUT_DIR, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla T4, 15360 MiB



In [6]:
# 2 · Execute 01 → 10. Notebooks that already PASSED (summary.json) are skipped, so after a fix just Run all again.
#     Set RERUN_ALL = True to start from scratch.
import re, time, json, nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

NOTEBOOKS = ["01_Screening_YOLO", "02_Screening_FasterRCNN", "03_Screening_Transformers", "04_Selection_and_Resolution",
             "05_Data_Scaling", "06_Tiling_Study", "07_Ground_Pretraining", "08_Final_Combination_and_Ablation",
             "09_Real_World_Transfer", "10_Final_Analysis"]
RERUN_ALL = False
PRELUDE = '''# [added by Run_Smoke_All] Drive is already mounted in this VM by the driver: reuse it, do not ask again
import os
from google.colab import drive as _drive
if os.path.isdir("/content/drive/MyDrive"):
    _drive.mount = lambda *a, **k: print("Drive already mounted by Run_Smoke_All")'''

summary_path = os.path.join(OUT_DIR, "summary.json")
summary = json.load(open(summary_path)) if os.path.exists(summary_path) and not RERUN_ALL else {}
for name in NOTEBOOKS:
    if summary.get(name, {}).get("ok"):
        print(f"✓ {name} already passed ({summary[name]['minutes']} min) — skipped", flush=True)
        continue
    nb = nbformat.read(os.path.join(NB_DIR, name + ".ipynb"), as_version=4)
    setup = next(c for c in nb.cells if c.cell_type == "code")
    setup.source, n = re.subn(r"^SMOKE_TEST = False", "SMOKE_TEST = True", setup.source, flags=re.M)
    assert n == 1, f"{name}: SMOKE_TEST line not found"
    nb.cells.insert(0, nbformat.v4.new_code_cell(PRELUDE))
    print(f"▶ {name} …", flush=True)
    t0, err = time.time(), None
    try:
        NotebookClient(nb, timeout=None, kernel_name="python3", resources={"metadata": {"path": "/content"}}).execute()
    except CellExecutionError as e:
        err = e
    nbformat.write(nb, os.path.join(OUT_DIR, name + ".ipynb"))
    mins = (time.time() - t0) / 60
    summary[name] = dict(ok=err is None, minutes=round(mins, 1), finished=time.strftime("%Y-%m-%d %H:%M"))
    json.dump(summary, open(summary_path, "w"), indent=1)
    print(f"{'PASS' if err is None else 'FAIL'} {name} in {mins:.1f} min", flush=True)
    if err:
        print(str(err)[-8000:])
        break
summary


✓ 01_Screening_YOLO already passed (4.7 min) — skipped
✓ 02_Screening_FasterRCNN already passed (0.7 min) — skipped
✓ 03_Screening_Transformers already passed (1.5 min) — skipped
✓ 04_Selection_and_Resolution already passed (1.9 min) — skipped
✓ 05_Data_Scaling already passed (0.9 min) — skipped
✓ 06_Tiling_Study already passed (5.2 min) — skipped
✓ 07_Ground_Pretraining already passed (1.4 min) — skipped
▶ 08_Final_Combination_and_Ablation …
PASS 08_Final_Combination_and_Ablation in 2.3 min
▶ 09_Real_World_Transfer …
PASS 09_Real_World_Transfer in 2.3 min
▶ 10_Final_Analysis …
PASS 10_Final_Analysis in 0.4 min


{'01_Screening_YOLO': {'ok': True,
  'minutes': 4.7,
  'finished': '2026-10-06 16:52'},
 '02_Screening_FasterRCNN': {'ok': True,
  'minutes': 0.7,
  'finished': '2026-10-06 16:53'},
 '03_Screening_Transformers': {'ok': True,
  'minutes': 1.5,
  'finished': '2026-10-06 16:54'},
 '04_Selection_and_Resolution': {'ok': True,
  'minutes': 1.9,
  'finished': '2026-10-06 16:56'},
 '05_Data_Scaling': {'ok': True,
  'minutes': 0.9,
  'finished': '2026-10-06 16:57'},
 '06_Tiling_Study': {'ok': True,
  'minutes': 5.2,
  'finished': '2026-10-06 17:02'},
 '07_Ground_Pretraining': {'ok': True,
  'minutes': 1.4,
  'finished': '2026-10-06 17:04'},
 '08_Final_Combination_and_Ablation': {'ok': True,
  'minutes': 2.3,
  'finished': '2026-10-06 17:14'},
 '09_Real_World_Transfer': {'ok': True,
  'minutes': 2.3,
  'finished': '2026-10-06 17:16'},
 '10_Final_Analysis': {'ok': True,
  'minutes': 0.4,
  'finished': '2026-10-06 17:17'}}

In [7]:
# 3 · Zip the executed copies for download (PDF records are made from these)
import shutil
from google.colab import files
z = shutil.make_archive("/content/smoke_executed", "zip", OUT_DIR)
files.download(z)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>